# Hornsby DCP 2024: build the PixelRAG index on a free Colab GPU

Your laptop needs about 8 hours for the 489 pages; a Colab GPU needs minutes. This notebook turns the PDF into a PixelRAG index and gives you back a **small zip (about 20 MB)** to unzip on your laptop.

**Before you start:** menu **Runtime > Change runtime type > T4 GPU**. Then run the cells top to bottom (Shift+Enter).

1. Cell 1 checks that a GPU is attached.
2. Cell 2 installs PixelRAG (about 2 minutes).
3. Cell 3 checks the install worked (this is the check that failed on the laptop until torchvision was fixed).
4. Cell 4 asks you to upload `hdcp-2024.pdf` (42 MB).
5. Cell 5 renders the pages. It starts in **TEST mode** (the same 10 pages tested on the laptop). Set `ALL = True` for the full 489 pages.
6. Cell 6 builds the index on the GPU.
7. Cell 7 downloads the small zip. Save it into `data\hornsby\` in the project.

In [ ]:
# 1. Is a GPU attached?
import subprocess
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip()
print(gpu or 'NO GPU FOUND: Runtime > Change runtime type > T4 GPU, then run this cell again.')

In [ ]:
# 2. Install PixelRAG (index extra) and PyMuPDF (to render the PDF pages)
!pip -q install "pixelrag[index]==0.4.0" pymupdf

In [ ]:
# 3. Check the install. torch and torchvision must match each other, otherwise the embed step fails with
#    'operator torchvision::nms does not exist' (the same trap we hit on the laptop).
import importlib, torch, torchvision
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| cuda available:', torch.cuda.is_available())
try:
    from torchvision.ops import nms
    from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
    print('OK: imports work')
except Exception as e:
    print('PROBLEM:', e)
    print('Fix: run the next cell, then Runtime > Restart session, then run cells 3 onwards again (skip cell 2).')

In [ ]:
# 3b. ONLY if cell 3 printed PROBLEM: reinstall torch and torchvision together so they match.
# !pip -q install --upgrade --force-reinstall torch torchvision

In [ ]:
# 4. Upload the Hornsby DCP PDF (data/hornsby/raw/hdcp-2024.pdf on your laptop, 42 MB)
from google.colab import files
uploaded = files.upload()
PDF = next(iter(uploaded))
print('got', PDF, round(len(uploaded[PDF]) / 1e6, 1), 'MB')

In [ ]:
# 5. Render pages to PNG at 150 dpi, named p0017.png etc. (the same naming and resolution as on the laptop,
#    so page numbers in search results are the real PDF page numbers).
import os, shutil, fitz

ALL = False   # False = TEST mode (10 pages, to compare with the laptop). True = all 489 pages.
TEST_PAGES = [17, 33, 37, 111, 177, 209, 244, 282, 294, 449]

doc = fitz.open(PDF)
pages = range(1, doc.page_count + 1) if ALL else TEST_PAGES
shutil.rmtree('pages', ignore_errors=True); os.makedirs('pages')
for p in pages:
    doc[p - 1].get_pixmap(dpi=150).save(f'pages/p{p:04d}.png')
print(len(os.listdir('pages')), 'pages rendered from', doc.page_count, '-page PDF; ALL =', ALL)

In [ ]:
# 6. Build the index on the GPU.
# IMPORTANT: --device auto (uses the GPU through PixelRAG's light embedder). --device cuda would switch to the
# vLLM/sglang backend, which is not installed here.
import time
t0 = time.time()
!pixelrag index build --source pages --source-type local --output index --device auto --force
print('index built in', round(time.time() - t0), 'seconds')

In [ ]:
# 7. Package ONLY the small files (search needs the vectors and page mapping, not the tile images) and download.
import pathlib, zipfile
OUT = 'hornsby_index_' + ('full' if ALL else 'test') + '.zip'
keep = {'index.faiss', 'metadata.npz', 'articles.json', 'summary.json', 'tiles.json', 'chunks.json'}
with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in pathlib.Path('index').rglob('*'):
        if f.is_file() and f.name in keep:
            z.write(f, f.relative_to('index'))
print(OUT, round(os.path.getsize(OUT) / 1e6, 1), 'MB')
files.download(OUT)